In [ ]:
# === Cell 1: Setup & repo clone =========================================
# Installs system deps, clones the repo, then installs Python deps in STAGES.
#
# Why stages and not `pip install -r requirements.txt`:
# NeMo and omnilingual-asr pin incompatible upper bounds on shared deps
# (transformers, fairseq2, lhotse, bitsandbytes). Asking pip to satisfy both
# in one resolution => ResolutionImpossible. Installing them one at a time
# lets each pull its own compatible set; last writer wins on shared deps.
# Per plan §3.2 / §8, an install failure here is non-fatal — the model that
# depends on the broken package gets marked FAILED later in Cell 4.

GITHUB_REPO_URL = "<<TODO>>"  # e.g. https://github.com/notAvailable73/stt-model-comparison.git
REPO_DIR = "/content/banglish-stt-benchmark"

import os, subprocess, sys

# libsndfile1 is needed by soundfile/librosa for reading wavs on Colab.
subprocess.run(["apt-get", "-qq", "install", "-y", "libsndfile1"], check=True)

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", GITHUB_REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("cwd:", os.getcwd())


def pip_install(args, label):
    """One install at a time so each gets its own resolution problem.
    check=False — a single stage failing is logged but doesn't crash setup."""
    print(f"\n--- installing: {label} ---")
    rc = subprocess.call([sys.executable, "-m", "pip", "install", "-q", *args])
    print(f"--- {label}: {'OK' if rc == 0 else f'FAILED rc={rc}'} ---")


# Stage 1: NeMo — heaviest dep tree (lhotse, lightning, downgrades transformers).
pip_install(["nemo_toolkit[asr]"], "NeMo  (models 4, 5, 6)")

# Stage 2: omnilingual-asr — pulls fairseq2. May further downgrade some shared
# deps; that's fine for our usage.
pip_install(["omnilingual-asr"], "omnilingual-asr  (model 7)")

# Stage 3: top up the libs WE call directly that may have been clobbered.
pip_install(
    ["openai", "pyyaml", "seaborn", "tqdm", "librosa", "soundfile",
     "requests", "pandas", "matplotlib"],
    "our own deps (top-up)",
)

# Quick probe so you can see ahead of Cell 4 which adapter families work.
print("\n--- adapter import check ---")
for label, mod in [
    ("transformers           (whisper_hf — models 1, 2, 3, 8)", "transformers"),
    ("nemo.collections.asr   (nemo_conformer — models 4, 5, 6)", "nemo.collections.asr"),
    ("omnilingual_asr        (omni_asr — model 7)", "omnilingual_asr"),
    ("openai                 (openrouter_stt — model 9)", "openai"),
]:
    try:
        __import__(mod)
        print(f"  OK    {label}")
    except Exception as e:
        print(f"  FAIL  {label} -- {e!r}")

print("\nSetup done. Any FAIL above means that adapter family's models will be marked FAILED in Cell 4 (per §3.2); the rest still run.")

In [ ]:
# === Cell 2: Secrets ====================================================
# OpenRouter API key — used by the Gemini 3 Flash judge AND the Chirp 3 STT model.
# Uses getpass so the key never gets logged in the notebook output.

import os
from getpass import getpass

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OPENROUTER_API_KEY: ")
print("OPENROUTER_API_KEY set:", bool(os.environ.get("OPENROUTER_API_KEY")))

# Optional: mount Drive for persistence across Colab sessions.
# from google.colab import drive
# drive.mount("/content/drive")

In [ ]:
# === Cell 3: Data preparation ===========================================
# Downloads OpenSLR-104 (skip if archive already there), extracts, parses
# transcripts, computes length + code-switch density per clip, stratified-
# samples 50 clips, writes data/manifest.csv, and prints 10 sample
# transcripts so you can verify the script (Bangla / Roman / mixed) per §2.4.

from src.utils import setup_logging, set_seeds
from src.data_prep import build_manifest, print_sample_transcripts

setup_logging("logs")
set_seeds(42)

manifest_path = build_manifest(
    raw_dir="data/raw",
    manifest_path="data/manifest.csv",
    n_samples=50,
    seed=42,
)
print_sample_transcripts(manifest_path, k=10, seed=0)

In [ ]:
# === Cell 4: Per-model transcription ====================================
# Runs every model in config/models.yaml over the 50 clips.
# - Predictions CSV per model under predictions/<slug>.csv
# - Flushed after every clip (resumable: rerun this cell to pick up where it stopped)
# - Load/run errors are logged; failed models go in `failed_models` and the
#   loop continues with the next one.

from src.transcribe import run_transcription

failed_models = run_transcription(
    manifest_csv="data/manifest.csv",
    models_yaml="config/models.yaml",
    predictions_dir="predictions",
    device="cuda",
)
print("\nFailed models:", failed_models if failed_models else "none")

In [ ]:
# === Cell 5: LLM judging ================================================
# One Gemini 3 Flash call per (clip, model). Resumable: rows already in
# judgments.csv are skipped, so reruns continue from the last judged pair.

from src.judge import judge_predictions

judge_predictions(
    manifest_csv="data/manifest.csv",
    predictions_dir="predictions",
    judgments_csv="judgments.csv",
    prompts_yaml="config/prompts.yaml",
)

In [ ]:
# === Cell 6: Build the master JSON ======================================
# Merges manifest + predictions + judgments into results/benchmark_results.json
# matching the schema in plan.md §6 Cell 6 exactly. Failed models go in
# metadata.models_failed (not into per-clip predictions).

from src.analyze import build_master_json

master_json = build_master_json(
    manifest_csv="data/manifest.csv",
    predictions_dir="predictions",
    judgments_csv="judgments.csv",
    models_yaml="config/models.yaml",
    failed_models=failed_models,
    output_json="results/benchmark_results.json",
)
print("Master JSON:", master_json)

In [ ]:
# === Cell 7: Comparison table ===========================================
# Per-model aggregates, sorted by Avg Overall desc. Two artifacts:
# - results/comparison_table.csv
# - results/comparison_table.png (styled matplotlib table)

from src.analyze import build_comparison_table
from IPython.display import Image, display
import pandas as pd

comp_csv, comp_png = build_comparison_table(
    manifest_csv="data/manifest.csv",
    predictions_dir="predictions",
    judgments_csv="judgments.csv",
    models_yaml="config/models.yaml",
    failed_models=failed_models,
    out_csv="results/comparison_table.csv",
    out_png="results/comparison_table.png",
)
display(pd.read_csv(comp_csv))
display(Image(str(comp_png)))

In [ ]:
# === Cell 8: Analytical plots ===========================================
# 6 PNGs in results/plots/ — see plan §6 Cell 8 for the full list.

from src.analyze import make_all_plots
from IPython.display import Image, display

plot_paths = make_all_plots(
    manifest_csv="data/manifest.csv",
    predictions_dir="predictions",
    judgments_csv="judgments.csv",
    models_yaml="config/models.yaml",
    failed_models=failed_models,
    plots_dir="results/plots",
)
for p in plot_paths:
    print(p)
    display(Image(str(p)))

In [ ]:
# === Cell 9: Final report (markdown summary) ============================
# Writes results/report.md and renders it inline.

from pathlib import Path
from src.analyze import write_report_md
from IPython.display import Markdown, display

report_path = write_report_md(
    comparison_csv="results/comparison_table.csv",
    master_json="results/benchmark_results.json",
    out_md="results/report.md",
)
display(Markdown(Path(report_path).read_text(encoding="utf-8")))

In [ ]:
# === Cell 10: Download bundle ===========================================
# Zips everything in results/ and triggers a browser download via
# google.colab.files. The zip contains the master JSON, the CSVs, all 6
# plots, the comparison-table PNG, and report.md.

import zipfile
from pathlib import Path
from src.analyze import bundle_results

zip_path = bundle_results("results", "banglish_stt_benchmark_results.zip")
size_mb = zip_path.stat().st_size / (1 << 20)
print(f"Bundle: {zip_path}  ({size_mb:.2f} MB)")
print("Contents:")
with zipfile.ZipFile(zip_path) as zf:
    for name in zf.namelist():
        print("  ", name)

try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("(google.colab not available — open", zip_path, "manually)")